# Crimes in Chicago - 2026
## Group 6:
Xinyi Chen, Zhongyin Wang

# Final Project, Part 2

The purpose of this assignment is to create a 'Viz for Experts' with an interactive dashboard interface for exploring your data.

For this submission option, you will submit your work through this Workspace.
    
**Please see Homework Prompt in PrairieLearn interface for more details on the requirements for this assignment.**

A rough outline of elements of code and write-up is shown below:

## Code:

 * An interactive dashboard within your Workspace that helps an expert explore your dataset thoroughly.
 * There should be a "dashboard" type aspect to this - i.e. a linked view exploring your dataset in an interactive way (like in Lab \#4) with [bqplot](https://bqplot.github.io/bqplot/).
 * Do not delete any cells, *just comment them out*. Show your work.



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt
import json
import urllib.request

In [ ]:
alt.data_transformers.disable_max_rows()

file_name = "Crimes_-_2026_20260417.csv"

# df_sample = df.sample(n=5000, random_state=42)
# The dataset isn’t very large, so it’s not necessary

try:
    df = pd.read_csv(file_name, low_memory=False)
    print("Data loaded successfully!")
    print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns\n")
    #display(df.head())
    
    df['Date'] = pd.to_datetime(df['Date'], format='%m/%d/%Y %I:%M:%S %p', errors='coerce')
    
except FileNotFoundError:
    print(f"Error: Could not find {file_name}. Please make sure it is in the same directory as this notebook.")

clean_geo_df = df.dropna(subset=['Longitude', 'Latitude']).copy()
print(f"\nRemaining rows after dropping missing coordinates: {clean_geo_df.shape[0]}")

In [ ]:
clean_geo_df['Date_Only'] = clean_geo_df['Date'].dt.floor('d')
clean_geo_df['District_Str'] = pd.to_numeric(clean_geo_df['District'], errors='coerce').fillna(-1).astype(int).astype(str)

brush = alt.selection_interval()
click_type = alt.selection_point(fields=['Primary Type'])
click_dist = alt.selection_point(fields=['District_Str'])

In [ ]:
geojson_url = 'https://data.cityofchicago.org/resource/24zt-jpfn.geojson'
response = urllib.request.urlopen(geojson_url)
chicago_geojson = json.loads(response.read())
districts = alt.Data(values=chicago_geojson['features'])

background = alt.Chart(districts).mark_geoshape(
    stroke='black',
    strokeWidth=0.6
).transform_calculate(
    District_Str="datum.properties.dist_num"
).encode(
    color=alt.condition(click_dist, alt.value('white'), alt.value('grey')),
    opacity=alt.condition(click_dist, alt.value(0.5), alt.value(0.8)),
    tooltip=[alt.Tooltip('properties.dist_num:N', title='District')]
).add_params(
    click_dist
)

geo_points = alt.Chart(clean_geo_df).mark_circle(size=5).encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    color=alt.condition(
        click_dist,
        alt.Color(
            'District:N',
            scale=alt.Scale(scheme='tableau10'),
            legend=alt.Legend(title='District', orient='right')
        ),
        alt.value('#e0dbd6')
    ),
    opacity=alt.condition(
        click_dist, 
        alt.value(0.6),
        alt.value(0.05)
    ),
    tooltip=['Primary Type', 'District', 'Date']
).add_params(
    brush
)

map_layer = (background + geo_points).project(type='mercator').properties(
    width=400,
    height=450,
    title="Chicago Crime Map (Brush / Click District)"
)

'''
geo_heatmap = alt.Chart(clean_geo_df).mark_rect().encode(
    x=alt.X('Longitude:Q', bin=alt.Bin(maxbins=60), title='Longitude'),
    y=alt.Y('Latitude:Q', bin=alt.Bin(maxbins=60), title='Latitude'),
    color=alt.Color('count():Q', scale=alt.Scale(scheme='reds'), title='Incident Count'),
    tooltip=['count()']
).properties(
    width=400,
    height=450,
    title="Chicago Crime Heatmap (Binned Grid)"
)

geo_heatmap
'''


In [ ]:
# plot2
type_chart = alt.Chart(clean_geo_df).mark_bar().encode(
    x=alt.X('count():Q', title='Number of Crimes'),
    y=alt.Y('Primary Type:N', sort='-x', title='Crime Type'),
    color=alt.condition(click_type, alt.value('steelblue'), alt.value('lightgray')),
    tooltip=['Primary Type', 'count()']
).properties(
    width=300,
    height=450,
    title='Crime Types'
).add_params(
    click_type
).transform_filter(
    brush
).transform_filter(
    click_dist
)

In [ ]:
df_period = clean_geo_df.copy()
df_period['Hour'] = df_period['Date'].dt.hour

def get_period(hour):
    if 6 < hour <= 12:
        return 'Morning (6am-12pm)'
    elif 12 < hour <= 18:
        return 'Afternoon (12pm-6pm)'
    elif 18 < hour <= 24:
        return 'Evening (6pm-12am)'
    else:
        return 'Late Night (12am-6am)'

df_period['Period'] = df_period['Hour'].apply(get_period)

# plot3
period_order = ['Morning (6am-12pm)', 'Afternoon (12pm-6pm)', 'Evening (6pm-12am)', 'Late Night (12am-6am)', 'Total Daily']
period_range = ['#f4a261', '#e9c46a', '#e76f51', '#264653', 'grey'] # 最后一个是 Total 的颜色

# Period
period_lines = alt.Chart(df_period).mark_line(point=False, strokeWidth=1.5).encode(
    x=alt.X('Date_Only:T', title='Timeline'),
    y=alt.Y('count:Q', title='Number of Incidents', scale=alt.Scale(zero=True)),
    color=alt.Color(
        'Period:N',
        scale=alt.Scale(domain=period_order, range=period_range), # 使用包含 Total 的 Scale
        legend=alt.Legend(title='Time of Day', orient='right')
    ),
    tooltip=[
        alt.Tooltip('Date_Only:T', title='Date'),
        alt.Tooltip('Period:N', title='Period'),
        alt.Tooltip('count:Q', title='Incidents')
    ]
).transform_filter(brush).transform_filter(click_type).transform_filter(click_dist).transform_aggregate(
    count='count()',
    groupby=['Date_Only', 'Period']
).transform_impute(
    impute='count',
    key='Date_Only',
    groupby=['Period'],
    value=0
)

# Total
total_line = alt.Chart(df_period).mark_line(
    opacity=0.5, 
    #strokeWidth=2
).encode(
    x=alt.X('Date_Only:T'),
    y=alt.Y('count():Q'),
    color=alt.datum('Total'), 
    tooltip=[
        alt.Tooltip('Date_Only:T', title='Date'),
        alt.Tooltip('count():Q', title='Total Incidents')
    ]
).transform_filter(brush).transform_filter(click_type).transform_filter(click_dist)

# Merge
line_chart = (total_line + period_lines).properties(
    width=750,
    height=220,
    title='Daily Crime Trend by Time of Day'
).resolve_scale(
    color='shared' 
)

line_chart

In [ ]:
# dashboard = (geo_heatmap | type_chart) & line_chart
dashboard = ((map_layer | type_chart) & line_chart).resolve_scale(color='independent')
dashboard

dashboard.save('interactive_dashboard.json')

## Prose:

* One paragraph explaining how to use the dashboard you created, to help someone who is not an expert understand your dataset.
* A list of 1 or more contextual datasets you have identified, links to where they reside, and a sentence about why they might be useful in telling the final story.
  * by "contextual dataset" here means a dataset that would add context to your chosen dataset. For example, if your dataset is the Champaign bus routes, some interesting contextual datasets could be the Chicago bus routes, or the Springfield bus routes, or the Amtrak routes in Champaign
  * you do not have to do anything with this dataset at the moment beyond writing a bit about why it would be useful. Looking forward, you will want to include "contextual visualizations" (which you may or may not generate on your own) in your Final Project, Part 3 and identifying a possibly useful dataset is a great way to start looking for contextual visualizations.
* If you have identified your dataset as a "large one" (i.e. larger than the GitHub file upload limit) comment on if you want to revise your plan for hosting this data or not. If this does not apply to your dataset please explicitly state this.
* Additionally, please note that as of writing, it is not possible to embed images within Starboard. Be sure to address how you plan on including your contextual dataset to add context to your main dataset given that you won't be able to directly embed images if you plan on using Starboard for Part 3.1 of the Final Project.


### Prose
This interactive dashboard allows us to visually explore reported crimes in Chicago based on location, type, and time. To use it, simply click on a specific police district on the top-left map, or drag mouse to draw a box over a neighborhood. This will automatically update the right-side bar chart to display the most common crimes in that specific area. We can also click on a specific crime category in the bar chart, which will dynamically update the bottom red line graph to show the timeline of those specific crimes, helping user easily identify temporal trends. 

#### *Contextual Dataset Identified:*
* *CTA - System Information - List of 'L' Stops:* https://data.cityofchicago.org/Transportation/CTA-System-Information-List-of-L-Stops/8pix-ypme

* *Why it's useful:* The "L" system is a major component of Chicago's infrastructure. By identifying station locations, we can eventually analyze whether certain crimes are centered around transit hubs, which provides essential context for understanding crime accessibility and patterns for our final story.

Since our dataset is not larger than the GitHub file upload limit, a revised data-hosting plan is not applicable. Additionally, to address Starboard's inability to embed static images for Part 3.1, we plan to include our contextual data programmatically, just as we loaded the Police District boundaries via `urllib` and JSON parsing in this dashboard, we will load future contextual geospatial data directly via URL to plot as native, interactive Altair layers.

## Plot Summary

Summarize the characteristics of the dataset in words: what does it represent, what are the fields/columns/rows, what data types are they, etc.

### Summary
This dataset represents a detailed log of reported crime incidents that occurred in the City of Chicago in the year 2026. Each row in the dataset corresponds to a single recorded crime incident. The dataset contains multiple fields (columns) encompassing different data types, which were utilized to build the visualizations:
* **Spatial Data:** `Latitude` and `Longitude` are numerical float values (Quantitative, `:Q`) used to plot the exact coordinates of incidents. `District` is a numerical code representing police jurisdictions, but it is treated as a categorical/string variable (Nominal, `:N`) for coloring and filtering.
* **Text Data:** Fields such as `Primary Type` describe the broad category of the crime (e.g., THEFT, BATTERY) and are treated as categorical strings (Nominal, `:N`). 
* **Temporal Data:** The `Date` field captures the exact timestamp of the incident. It was parsed into a Pandas datetime object (`Date_Only`) and treated as temporal data (Temporal, `:T`) to build the time-series line chart.
* **Aggregated Data:** The dashboard relies heavily on aggregating the number of rows (`count():Q`) to display incident frequencies across the different charts based on user interactions.

## Final Project, Part 3
In this part, we iterated our Part 2 dashboard, making it more effective to connect to Part 3 story.

In [ ]:
cta_url = "https://data.cityofchicago.org/resource/8pix-ypme.json"
df_cta = pd.read_json(cta_url)

df_cta['lat'] = df_cta['location'].apply(lambda x: float(x['latitude']) if x else None)
df_cta['lon'] = df_cta['location'].apply(lambda x: float(x['longitude']) if x else None)

cta_map = alt.Chart(df_cta).mark_circle(size=20, color='orange', opacity=0.7).encode(
    longitude='lon:Q',
    latitude='lat:Q',
    tooltip=[
        alt.Tooltip('station_name:N', title='Station Name'),
        alt.Tooltip('lines:N', title='Lines Served')
    ]
).properties(
    title='Chicago "L" Station Locations',
    width=600,
    height=400
)

display(cta_map)
#cta_map.save('cta_locations.json')

In [ ]:
df['Month'] = df['Date'].dt.month

crime_trend = alt.Chart(df).mark_bar(color='steelblue').encode(
    x=alt.X('Month:O', title='Month'),
    y=alt.Y('count():Q', title='Total Number of Crimes'),
    tooltip=[alt.Tooltip('Month:O'), alt.Tooltip('count():Q', title='Total Crimes')]
).properties(
    title='Monthly Crime Trends in Chicago',
    width=600,
    height=300
)

display(crime_trend)
crime_trend.save('crime_annual_trends.json')

In [ ]:
# === Contextual Visualization 1: CTA 'L' Stations Overlaid on Crime Map ===
# This shows whether crimes cluster near transit hubs.

cta_url = "https://data.cityofchicago.org/resource/8pix-ypme.json"
df_cta = pd.read_json(cta_url)

df_cta['lat'] = df_cta['location'].apply(lambda x: float(x['latitude']) if isinstance(x, dict) else None)
df_cta['lon'] = df_cta['location'].apply(lambda x: float(x['longitude']) if isinstance(x, dict) else None)
df_cta = df_cta.dropna(subset=['lat', 'lon'])

# --- Layer 1: District boundaries (base) ---
cta_background = alt.Chart(districts).mark_geoshape(
    fill='#f0f0f0', stroke='#aaa', strokeWidth=0.5
)

# --- Layer 2: Crime incidents (red dots) ---
crime_heatmap_layer = alt.Chart(clean_geo_df).mark_circle(
    size=4, opacity=0.25, color='crimson'
).encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q',
    tooltip=['Primary Type:N', 'District:N']
)

# --- Layer 3: CTA station overlay (orange circles) ---
cta_layer = alt.Chart(df_cta).mark_circle(
    size=40, color='orange', opacity=0.85, stroke='white', strokeWidth=0.8
).encode(
    longitude='lon:Q',
    latitude='lat:Q',
    tooltip=[alt.Tooltip('station_name:N', title='Station')]
)

cta_overlay_map = (cta_background + crime_heatmap_layer + cta_layer).project(
    type='mercator'
).properties(
    width=600,
    height=500,
    title=alt.TitleParams(
        text="Chicago Crime Incidents vs. CTA 'L' Station Locations",
        subtitle="Red dots = crime incidents  |  Orange circles = 'L' stations",
        fontSize=14
    )
)

display(cta_overlay_map)
# cta_overlay_map.save('cta_crime_overlay.json')

In [ ]:
import urllib.request, json

community_url = 'https://data.cityofchicago.org/resource/igwz-8jzy.geojson'
response = urllib.request.urlopen(community_url)
community_geojson = json.loads(response.read())
communities = alt.Data(values=community_geojson['features'])

socio_url = "https://data.cityofchicago.org/resource/kn9c-c2s2.json"
df_socio = pd.read_json(socio_url)

'''
df_socio['ca'] = df_socio['ca'].astype(float).astype(int).astype(str)
df_socio['poverty_rate'] = pd.to_numeric(
    df_socio['percent_households_below_poverty'], errors='coerce'
)
'''
df_socio = df_socio.dropna(subset=['ca'])
df_socio['ca'] = df_socio['ca'].astype(float).astype(int).astype(str)
df_socio['poverty_rate'] = pd.to_numeric(
    df_socio['percent_households_below_poverty'], errors='coerce'
)

'''
# community_area_number, community_area_name, percent_households_below_poverty
df_socio['ca'] = df_socio['community_area_number'].astype(str)
df_socio['poverty_rate'] = pd.to_numeric(
    df_socio['percent_households_below_poverty'], errors='coerce'
)
'''

In [ ]:
# === Contextual Visualization Poverty Rate Choropleth ===

poverty_choropleth = alt.Chart(communities).mark_geoshape(
    stroke='white', strokeWidth=0.4
).transform_lookup(
    lookup='properties.area_num_1',
    from_=alt.LookupData(df_socio, 'ca', ['poverty_rate', 'community_area_name'])
).encode(
    color=alt.Color(
        'poverty_rate:Q',
        scale=alt.Scale(scheme='orangered'),
        title='Poverty Rate (%)'
    ),
    tooltip=[
        alt.Tooltip('properties.community:N', title='Community'),
        alt.Tooltip('poverty_rate:Q', title='Poverty Rate (%)', format='.1f')
    ]
).project(type='mercator').properties(
    width=380,
    height=460,
    title='Chicago Poverty Rate by Community Area'
)

crime_dots = alt.Chart(clean_geo_df.sample(5000, random_state=42)).mark_circle(
    size=3, color='steelblue', opacity=0.3
).encode(
    longitude='Longitude:Q',
    latitude='Latitude:Q'
)

display(poverty_choropleth + crime_dots)
(poverty_choropleth + crime_dots).save('poverty_crime_overlay.json')

In [ ]:
print(clean_geo_df['Community Area'].dtype)
print(clean_geo_df['Community Area'].dropna().head(10))
print(df_socio['ca'].head(10))

In [ ]:
df_crime_count = clean_geo_df.groupby('Community Area').size().reset_index(name='crime_count')
df_crime_count['ca'] = df_crime_count['Community Area'].astype(int).astype(str)

df_scatter = pd.merge(df_socio[['ca', 'community_area_name', 'poverty_rate']], 
                      df_crime_count, on='ca', how='inner')

print(f"Merged rows: {len(df_scatter)}")
print(df_scatter.sort_values('crime_count', ascending=False).head(5))

In [ ]:
scatter = alt.Chart(df_scatter).mark_circle(size=80, opacity=0.75).encode(
    x=alt.X('poverty_rate:Q', title='Poverty Rate (%)'),
    y=alt.Y('crime_count:Q', title='Number of Crimes (2026)'),
    color=alt.Color('poverty_rate:Q', scale=alt.Scale(scheme='orangered'), legend=None),
    tooltip=[
        alt.Tooltip('community_area_name:N', title='Community'),
        alt.Tooltip('poverty_rate:Q', title='Poverty Rate (%)', format='.1f'),
        alt.Tooltip('crime_count:Q', title='Crime Count')
    ]
)

regression = scatter.transform_regression(
    'poverty_rate', 'crime_count'
).mark_line(color='gray', strokeDash=[4, 4], strokeWidth=1.5)

display(
    (scatter + regression).properties(
        width=550,
        height=350,
        title=alt.TitleParams(
            text='Higher Poverty, More Crimes? A Community-Level View',
            subtitle='Each dot = one Chicago community area  |  Dashed line = trend',
            fontSize=14
        )
    )
)
(scatter + regression).save('poverty_scatter_regression.json')

In [ ]:
df_time = clean_geo_df.copy()
df_time['Hour'] = df_time['Date'].dt.hour
df_time['Weekday'] = df_time['Date'].dt.dayofweek

weekday_map = {0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}
weekday_order = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
df_time['Weekday_Name'] = df_time['Weekday'].map(weekday_map)

# Select top 10 crime types as sample
top_types = df_time['Primary Type'].value_counts().head(10).index.tolist()
df_time_filtered = df_time[df_time['Primary Type'].isin(top_types)]

crime_select = alt.binding_select(
    options=['All'] + top_types,
    name='Crime Type: '
)
crime_param = alt.param(name='crime_type', value='All', bind=crime_select)

heatmap = alt.Chart(df_time_filtered).mark_rect().transform_filter(
    "(crime_type === 'All') || (datum['Primary Type'] === crime_type)"
).encode(
    x=alt.X('Weekday_Name:N', sort=weekday_order, title='Day of Week'),
    y=alt.Y('Hour:O', title='Hour of Day', sort='ascending'),
    color=alt.Color(
        'count():Q',
        scale=alt.Scale(scheme='reds'),
        title='Number of Crimes'
    ),
    tooltip=[
        alt.Tooltip('Weekday_Name:N', title='Day'),
        alt.Tooltip('Hour:O', title='Hour'),
        alt.Tooltip('count():Q', title='Total Crimes'),
    ]
).add_params(
    crime_param
).properties(
    width=500,
    height=400,
    title=alt.TitleParams(
        text='When Do Crimes Happen in Chicago?',
        subtitle='Select a crime type from the dropdown to filter',
        fontSize=14
    )
)

display(heatmap)
heatmap.save('crime_heatmap_interactive.json')